In [3]:
# UCBT modeling 

import os
import json
import warnings
import joblib
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter
from pandas.api.types import is_numeric_dtype
from IPython.display import display

from sklearn.model_selection import (
    GroupShuffleSplit, StratifiedKFold, train_test_split, cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline as SkPipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, roc_curve
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# SHAP 
try:
    import shap
    HAS_SHAP = True
    warnings.filterwarnings("ignore", category=UserWarning, module="shap")
except Exception:
    HAS_SHAP = False
    print("[WARN] SHAP unavailable; skipping explainability steps.")


# Config & paths
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE = Path("/Users/amanda/Desktop/UCBT")

# Choose which dataset to run:
# Uncoment ONE of the following lines per run
# Synthetic dataset (used for Approach 2)
DATA_PATH = BASE / "ucbt_dataset_synthetic_best.csv"   # synthetic
# Real dataset (used for Approach 1)
#DATA_PATH = BASE / "ucbt_dataset.csv" # real 
OUTDIR_NAME = DATA_PATH.stem.replace("ucbt_dataset_", "") 
OUTDIR = BASE / f"models_output_{OUTDIR_NAME}"
(OUTDIR / "metrics").mkdir(parents=True, exist_ok=True)
(OUTDIR / "models").mkdir(parents=True, exist_ok=True)
(OUTDIR / "shap").mkdir(parents=True, exist_ok=True)
(OUTDIR / "figs").mkdir(parents=True, exist_ok=True)
print("Using dataset:", DATA_PATH)
print("Using folder:", OUTDIR)


# Features/outcomes

MODEL_FEATURES = [
    "Recipient_Age", "Recipient_Sex", "Cord_Blood_Units", "Remission_Status",
    "CD34_num", "TNC_num", "hla_best", "hla_worst", "hla_any6", "hla_double",
    "HLAxCD34", "HLAxTNC", "Age_x_CD34", "Age_x_TNC", "log_CD34_num", "log_TNC_num",
    "Age2", "hla_mean", "hla_range", "CD34_adequate", "TNC_adequate", "Adequacy_Both",
    "Regimen_MA", "Age_x_RegimenMA", "CD34_num_q_by_DzHLA", "TNC_num_q_by_DzHLA"
]
BASE_CATS = ["Ethnicity", "Race", "Disease_Type", "Conditioning_Regimen", "HLA_Match_Level", "AgeBin"]
OUTCOMES = ["Neutrophil_Engraftment", "Platelet_Engraftment", "Chronic_GVHD", "1_Year_Survival"]

# OneHotEncoder config 
cat_encoder_kwargs = {"handle_unknown": "ignore"}
if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    cat_encoder_kwargs["sparse_output"] = False
else:
    cat_encoder_kwargs["sparse"] = False


# Helpers
def best_threshold_for_roc_auc(y_true, proba, lo=0.20, hi=0.80, steps=61):
    grid = np.linspace(lo, hi, steps)
    aucs = [roc_auc_score(y_true, (proba >= t).astype(int)) for t in grid]
    return float(grid[np.argmax(aucs)])

class AlignTwoCols:
    """Rank-align CD34_num and TNC_num."""
    def __init__(self, cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99):
        self.cols = list(cols)
        self.q_lo = q_lo
        self.q_hi = q_hi
        self.params_ = {}

    def fit(self, df):
        self.params_.clear()
        for c in self.cols:
            if c not in df.columns:
                continue
            x = pd.to_numeric(df[c], errors="coerce").dropna().values
            if x.size == 0:
                continue
            ql, qh = np.quantile(x, [self.q_lo, self.q_hi])
            xs = np.sort(np.clip(x, ql, qh))
            self.params_[c] = {"ql": float(ql), "qh": float(qh), "sorted": xs}
        return self

    def transform(self, df):
        out = df.copy()
        for c, p in self.params_.items():
            if c not in out.columns:
                continue
            x = pd.to_numeric(out[c], errors="coerce").astype(float).clip(p["ql"], p["qh"])
            xs = p["sorted"]
            if xs.size <= 1:
                out[f"{c}__aligned"] = x
            else:
                ranks = np.searchsorted(xs, x, side="right") / xs.size
                out[f"{c}__aligned"] = ranks
        return out

    def fit_transform(self, df):
        return self.fit(df).transform(df)

def build_pipeline(preprocessor, model, use_smote: bool, model_name: str):
    """
    SMOTE is applied only for non-XGB models AND only when requested (use_smote).
    Preprocessor is a ColumnTransformer that returns a dense matrix,
    so SMOTE can consume it safely.
    """
    steps = [("preprocess", preprocessor)]
    if model_name != "xgb" and use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE, sampling_strategy="auto")))
    steps.append(("clf", model))
    return ImbPipeline(steps)

def make_model(model_name, outcome):
    if model_name == "xgb":
        return XGBClassifier(
            n_estimators=1200, max_depth=4, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=2,
            reg_lambda=1.0, reg_alpha=0.0, n_jobs=-1, eval_metric="logloss",
            tree_method="hist", random_state=RANDOM_STATE
        )
    elif model_name == "gbm":
        return HistGradientBoostingClassifier(
            max_depth=4, learning_rate=0.05, max_iter=800,
            early_stopping=True, validation_fraction=0.15,
            random_state=RANDOM_STATE
        )
    elif model_name == "rf":
        return RandomForestClassifier(
            n_estimators=1200 if outcome in ("Neutrophil_Engraftment", "Platelet_Engraftment") else 1000,
            max_depth=16 if outcome in ("Neutrophil_Engraftment", "Platelet_Engraftment") else None,
            min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
        )
    elif model_name == "svm":
        return SVC(kernel="rbf", C=1.0, probability=True, random_state=RANDOM_STATE)
    raise ValueError(f"Unsupported model: {model_name}")

def plot_roc(pipe, X, y, title, path):
    prob = pipe.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, prob)
    auc = roc_auc_score(y, prob)
    plt.figure(figsize=(5.5, 4.5))
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(title); plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=220, bbox_inches="tight"); plt.close()


# Load & prepare
df = pd.read_csv(DATA_PATH)
drop_cols = [c for c in df.columns if c.endswith("_pre")]
df = df.drop(columns=drop_cols, errors="ignore")
feature_pool = [c for c in (MODEL_FEATURES + BASE_CATS) if c in df.columns]

# Train/Eval loop
all_rows = []
HOLDOUTS = {}
models_to_run = ["xgb", "gbm", "rf", "svm"]

for outcome in OUTCOMES:
    if outcome not in df.columns:
        print(f"Skipping {outcome}: column missing in data")
        continue

    y = df[outcome]
    if y.notna().sum() < 100 or y.nunique() < 2:
        print(f"Skipping {outcome}: insufficient data or single class")
        continue

    X = df[feature_pool].copy()
    y = y.astype(int)

    # Try both 70/30 and 80/20 splits
    for split_name, test_size in [("70_30", 0.30), ("80_20", 0.20)]:
        # Grouped split if Study_ID exists
        if "Study_ID" in df.columns:
            gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)
            tr_pos, ho_pos = next(gss.split(X, y, groups=df["Study_ID"]))
            tr_idx, ho_idx = X.index[tr_pos], X.index[ho_pos]
        else:
            tr_idx, ho_idx = train_test_split(
                X.index, test_size=test_size, random_state=RANDOM_STATE, stratify=y
            )
        HOLDOUTS[f"{outcome}_{split_name}"] = (tr_idx, ho_idx)

        X_tr, X_te = X.loc[tr_idx].copy(), X.loc[ho_idx].copy()
        y_tr, y_te = y.loc[tr_idx].copy(), y.loc[ho_idx].copy()

        # Robust align CD34/TNC
        aligner = AlignTwoCols(cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99)
        X_tr = aligner.fit_transform(X_tr)
        X_te = aligner.transform(X_te)

        # Build column lists (after aligner adds *_aligned)
        num_cols = [c for c in X_tr.columns if is_numeric_dtype(X_tr[c])]
        cat_cols = [c for c in X_tr.columns if not is_numeric_dtype(X_tr[c])]

        # impute -> scale for numeric; impute -> OHE for categorical
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", SkPipeline([
                    ("imp", SimpleImputer(strategy="median")),
                    ("sc", MinMaxScaler())
                ]), num_cols),
                ("cat", SkPipeline([
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(**cat_encoder_kwargs))
                ]), cat_cols),
            ],
            remainder="drop",
            verbose_feature_names_out=False,
        )

        # Decide if SMOTE is needed (class imbalance on y_tr)
        cnt = Counter(y_tr)
        mn, mx = min(cnt.values()), max(cnt.values())
        ratio = mn / mx if mx else 0.0
        USE_SMOTE = (ratio < 0.80 and mn >= 6)  # avoid pathological resampling

        for model_name in models_to_run:
            model = make_model(model_name, outcome)
            pipe = build_pipeline(preprocessor, model, use_smote=USE_SMOTE, model_name=model_name)

            # 3-fold CV
            cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
            cv_scores = cross_validate(
                pipe, X_tr, y_tr, cv=cv,
                scoring=['roc_auc', 'accuracy', 'precision', 'recall'],
                return_train_score=False, n_jobs=-1
            )

            # Threshold tuning on a small validation slice from train
            tr_i, va_i = train_test_split(
                X_tr.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y_tr
            )
            pipe.fit(X_tr.loc[tr_i], y_tr.loc[tr_i])
            val_proba = pipe.predict_proba(X_tr.loc[va_i])[:, 1]
            t_star = best_threshold_for_roc_auc(y_tr.loc[va_i], val_proba)

            # Full training and test evaluation
            pipe.fit(X_tr, y_tr)
            proba_te = pipe.predict_proba(X_te)[:, 1]
            pred_te = (proba_te >= t_star).astype(int)

            auc_te = roc_auc_score(y_te, proba_te)
            acc_te = accuracy_score(y_te, pred_te)
            prec_te = precision_score(y_te, pred_te, zero_division=0)
            rec_te = recall_score(y_te, pred_te, zero_division=0)

            # Save model+threshold for 70/30 split (the primary one)
            if split_name == "70_30":
                model_path = OUTDIR / "models" / f"{model_name}_{outcome}.joblib"
                joblib.dump(pipe, model_path)
                threshold_path = OUTDIR / "models" / f"{model_name}_{outcome}_threshold.json"
                with open(threshold_path, "w") as f:
                    json.dump({"threshold": float(t_star)}, f)
                print(f"Saved model for {model_name} - {outcome} -> {model_path}")
                print(f"Saved threshold for {model_name} - {outcome} -> {threshold_path}")

            smote_applied = (model_name != "xgb" and USE_SMOTE)
            row = {
                "outcome": outcome,
                "split": split_name,
                "model": model_name,
                "cv_auc_mean": float(cv_scores['test_roc_auc'].mean()),
                "cv_auc_std": float(cv_scores['test_roc_auc'].std()),
                "cv_acc_mean": float(cv_scores['test_accuracy'].mean()),
                "test_auc": auc_te,
                "test_acc": acc_te,
                "test_prec": prec_te,
                "test_rec": rec_te,
                "used_smote": smote_applied,
                "auc_pass": auc_te >= 0.80
            }
            all_rows.append(row)

            print(f"[Split {split_name}] {model_name} - {outcome}: "
                  f"test_auc={auc_te:.3f}, cv_auc_mean={row['cv_auc_mean']:.3f}, SMOTE={smote_applied}")


# Save CV summary
cv_table = pd.DataFrame(all_rows).sort_values(
    ["outcome", "split", "model", "test_auc"],
    ascending=[True, True, True, False]
)
cv_table['auc_flag'] = np.where(cv_table['test_auc'] < 0.80, "FAIL", "PASS")
print("Validation Summary (3-fold CV, 70/30 and 80/20 splits):")
display(cv_table)

cv_path = OUTDIR / "metrics" / "cv_summary.csv"
cv_table.to_csv(cv_path, index=False)
print(f"Saved CV summary -> {cv_path}")

# Best models (70/30)
best_models = cv_table[cv_table['split'] == "70_30"].groupby('outcome').apply(
    lambda x: x.loc[x['test_auc'].idxmax()]
).reset_index(drop=True)

print("Best Models per Outcome (70/30 split):")
display(best_models[['outcome', 'model', 'test_auc', 'auc_flag', 'used_smote']])
best_models.to_csv(OUTDIR / "metrics" / "best_models.csv", index=False)
print(f"Saved best models -> {OUTDIR / 'metrics' / 'best_models.csv'}")

# SHAP (tree models)
if HAS_SHAP:
    for outcome in OUTCOMES:
        if outcome not in df.columns:
            continue
        y = df[outcome]
        if y.notna().sum() < 100 or y.nunique() < 2:
            continue

        X = df[feature_pool].copy()
        y = y.astype(int)

        for model_name in ["xgb", "gbm", "rf"]:  # SVM not supported by TreeExplainer
            model_path = OUTDIR / "models" / f"{model_name}_{outcome}_{OUTDIR_NAME}.joblib"
            if not model_path.exists():
                continue

            pipe = joblib.load(model_path)

            # Align then transform with the stored preprocessor
            aligner = AlignTwoCols(cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99)
            X = aligner.fit_transform(X)

            # Transform via preprocessor
            X_trans = pipe.named_steps["preprocess"].transform(X)
            X_dense = X_trans.toarray() if hasattr(X_trans, "toarray") else np.asarray(X_trans)

            try:
                feat_names = pipe.named_steps["preprocess"].get_feature_names_out()
            except Exception:
                # Fallback feature names
                n_cols = X_dense.shape[1]
                feat_names = [f"f{i}" for i in range(n_cols)]

            # Subsample for SHAP speed
            sample = min(2000, X_dense.shape[0])
            idx = np.random.choice(np.arange(X_dense.shape[0]), size=sample, replace=False)
            X_sample = X_dense[idx]

            try:
                explainer = shap.TreeExplainer(pipe.named_steps["clf"])
                shap_values = explainer.shap_values(X_sample)
                sv = shap_values[1] if isinstance(shap_values, list) and len(shap_values) > 1 else shap_values

                plt.figure()
                shap.summary_plot(sv, X_sample, feature_names=feat_names, show=False, max_display=20)
                out_png = OUTDIR / "shap" / f"shap_summary_{model_name}_{outcome}.png"
                plt.tight_layout(); plt.savefig(out_png, dpi=220, bbox_inches="tight"); plt.close()
                print(f"Saved SHAP summary -> {out_png}")

                plt.figure()
                shap.summary_plot(sv, X_sample, feature_names=feat_names, plot_type="bar", show=False, max_display=20)
                out_png2 = OUTDIR / "shap" / f"shap_bar_{model_name}_{outcome}.png"
                plt.tight_layout(); plt.savefig(out_png2, dpi=220, bbox_inches="tight"); plt.close()
                print(f"Saved SHAP bar -> {out_png2}")

                vals = np.abs(sv).mean(axis=0)
                top_k = min(25, len(vals))
                top_idx = np.argsort(-vals)[:top_k]
                top_df = pd.DataFrame({"feature": np.array(feat_names)[top_idx], "mean_abs_shap": vals[top_idx]})
                top_csv = OUTDIR / "shap" / f"top_features_{model_name}_{outcome}.csv"
                top_df.to_csv(top_csv, index=False)
                print(f"Saved top-features CSV -> {top_csv}")
            except Exception as e:
                print(f"[WARN] SHAP failed for {model_name} - {outcome}: {e}")


# ROC curves
for outcome in OUTCOMES:
    if outcome not in df.columns:
        continue
    y = df[outcome]
    if y.notna().sum() < 100 or y.nunique() < 2:
        continue

    X = df[feature_pool].copy()
    y = y.astype(int)
    aligner = AlignTwoCols(cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99)
    X = aligner.fit_transform(X)

    for model_name in models_to_run:
        model_path = OUTDIR / "models" / f"{model_name}_{outcome}.joblib"
        if not model_path.exists():
            continue
        pipe = joblib.load(model_path)
        out = OUTDIR / "figs" / f"roc_{model_name}_{outcome}.png"
        try:
            plot_roc(pipe, X, y, f"ROC: {model_name.upper()} - {outcome}", out)
            print(f"Saved ROC -> {out}")
        except Exception as e:
            print(f"[WARN] ROC plotting failed for {model_name} - {outcome}: {e}")


# Run metadata
meta = {
    "dataset": str(DATA_PATH),
    "rows": int(df.shape[0]),
    "random_state": RANDOM_STATE,
    "saved_models": [
        str(OUTDIR / "models" / f"{m}_{o}.joblib")
        for o in OUTCOMES for m in models_to_run
        if (OUTDIR / "models" / f"{m}_{o}.joblib").exists()
    ]
}
with open(OUTDIR / "run_info.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"Saved run info -> {OUTDIR / 'run_info.json'}")
print("End of script.")


Using dataset: /Users/amanda/Desktop/UCBT/ucbt_dataset_synthetic_best.csv
Using folder: /Users/amanda/Desktop/UCBT/models_output_synthetic_best
Saved model for xgb - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Neutrophil_Engraftment.joblib
Saved threshold for xgb - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Neutrophil_Engraftment_threshold.json
[Split 70_30] xgb - Neutrophil_Engraftment: test_auc=0.554, cv_auc_mean=0.606, SMOTE=False
Saved model for gbm - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Neutrophil_Engraftment.joblib
Saved threshold for gbm - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Neutrophil_Engraftment_threshold.json
[Split 70_30] gbm - Neutrophil_Engraftment: test_auc=0.560, cv_auc_mean=0.610, SMOTE=True
Saved model for rf - Neutrophil_Engraftment -> /Users/amanda/Desktop/UC

,outcome,split,model,cv_auc_mean,cv_auc_std,cv_acc_mean,test_auc,test_acc,test_prec,test_rec,used_smote,auc_pass,auc_flag
25,1_Year_Survival,70_30,gbm,0.553838,0.002384,0.563268,0.546061,0.537723,0.539515,0.770214,True,False,FAIL
26,1_Year_Survival,70_30,rf,0.555391,0.002873,0.560448,0.549222,0.537331,0.553791,0.576909,True,False,FAIL
27,1_Year_Survival,70_30,svm,0.560700,0.001895,0.557428,0.529113,0.520870,0.529673,0.718315,True,False,FAIL
24,1_Year_Survival,70_30,xgb,0.548890,0.001750,0.562193,0.542319,0.519302,0.554555,0.393757,False,False,FAIL
29,1_Year_Survival,80_20,gbm,0.561000,0.004899,0.560819,0.533256,0.529062,0.560606,0.622372,True,False,FAIL
30,1_Year_Survival,80_20,rf,0.558231,0.001885,0.555487,0.545891,0.531808,0.575730,0.530698,True,False,FAIL
31,1_Year_Survival,80_20,svm,0.565077,0.001530,0.555712,0.530044,0.521739,0.563717,0.535744,True,False,FAIL
28,1_Year_Survival,80_20,xgb,0.557717,0.001380,0.559304,0.536895,0.531808,0.565149,0.605551,False,False,FAIL
17,Chronic_GVHD,70_30,gbm,0.562301,0.002035,0.747332,0.572210,0.647462,0.236387,0.378074,True,False,FAIL
18,Chronic_GVHD,70_30,rf,0.557362,0.008374,0.729208,0.547900,0.726631,0.239103,0.196721,True,False,FAIL


Saved CV summary -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/metrics/cv_summary.csv
Best Models per Outcome (70/30 split):


/var/folders/3b/hnsfxtfj2w39h1fm21zknsf40000gn/T/ipykernel_35043/2517871401.py:322: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_models = cv_table[cv_table['split'] == "70_30"].groupby('outcome').apply(


,outcome,model,test_auc,auc_flag,used_smote
0,1_Year_Survival,rf,0.549222,FAIL,True
1,Chronic_GVHD,svm,0.580646,FAIL,True
2,Neutrophil_Engraftment,gbm,0.560360,FAIL,True
3,Platelet_Engraftment,svm,0.552954,FAIL,True


Saved best models -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/metrics/best_models.csv
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_xgb_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_gbm_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_rf_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_svm_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_xgb_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_gbm_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_rf_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_svm_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UC

In [ ]:
what is this warning?
Using dataset: /Users/amanda/Desktop/UCBT/ucbt_dataset_synthetic_best.csv
Using folder: /Users/amanda/Desktop/UCBT/models_output_synthetic_best
Saved model for xgb - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Neutrophil_Engraftment.joblib
Saved threshold for xgb - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Neutrophil_Engraftment_threshold.json
[Split 70_30] xgb - Neutrophil_Engraftment: test_auc=0.554, cv_auc_mean=0.606, SMOTE=False
Saved model for gbm - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Neutrophil_Engraftment.joblib
Saved threshold for gbm - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Neutrophil_Engraftment_threshold.json
[Split 70_30] gbm - Neutrophil_Engraftment: test_auc=0.560, cv_auc_mean=0.610, SMOTE=True
Saved model for rf - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_Neutrophil_Engraftment.joblib
Saved threshold for rf - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_Neutrophil_Engraftment_threshold.json
[Split 70_30] rf - Neutrophil_Engraftment: test_auc=0.552, cv_auc_mean=0.598, SMOTE=True
Saved model for svm - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_Neutrophil_Engraftment.joblib
Saved threshold for svm - Neutrophil_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_Neutrophil_Engraftment_threshold.json
[Split 70_30] svm - Neutrophil_Engraftment: test_auc=0.545, cv_auc_mean=0.602, SMOTE=True
[Split 80_20] xgb - Neutrophil_Engraftment: test_auc=0.577, cv_auc_mean=0.607, SMOTE=False
[Split 80_20] gbm - Neutrophil_Engraftment: test_auc=0.579, cv_auc_mean=0.601, SMOTE=True
[Split 80_20] rf - Neutrophil_Engraftment: test_auc=0.552, cv_auc_mean=0.594, SMOTE=True
[Split 80_20] svm - Neutrophil_Engraftment: test_auc=0.539, cv_auc_mean=0.601, SMOTE=True
Saved model for xgb - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Platelet_Engraftment.joblib
Saved threshold for xgb - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Platelet_Engraftment_threshold.json
[Split 70_30] xgb - Platelet_Engraftment: test_auc=0.527, cv_auc_mean=0.513, SMOTE=False
Saved model for gbm - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Platelet_Engraftment.joblib
Saved threshold for gbm - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Platelet_Engraftment_threshold.json
[Split 70_30] gbm - Platelet_Engraftment: test_auc=0.544, cv_auc_mean=0.511, SMOTE=True
Saved model for rf - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_Platelet_Engraftment.joblib
Saved threshold for rf - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_Platelet_Engraftment_threshold.json
[Split 70_30] rf - Platelet_Engraftment: test_auc=0.529, cv_auc_mean=0.502, SMOTE=True
Saved model for svm - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_Platelet_Engraftment.joblib
Saved threshold for svm - Platelet_Engraftment -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_Platelet_Engraftment_threshold.json
[Split 70_30] svm - Platelet_Engraftment: test_auc=0.553, cv_auc_mean=0.516, SMOTE=True
[Split 80_20] xgb - Platelet_Engraftment: test_auc=0.541, cv_auc_mean=0.525, SMOTE=False
[Split 80_20] gbm - Platelet_Engraftment: test_auc=0.549, cv_auc_mean=0.519, SMOTE=True
[Split 80_20] rf - Platelet_Engraftment: test_auc=0.547, cv_auc_mean=0.509, SMOTE=True
[Split 80_20] svm - Platelet_Engraftment: test_auc=0.561, cv_auc_mean=0.517, SMOTE=True
Saved model for xgb - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Chronic_GVHD.joblib
Saved threshold for xgb - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_Chronic_GVHD_threshold.json
[Split 70_30] xgb - Chronic_GVHD: test_auc=0.567, cv_auc_mean=0.557, SMOTE=False
Saved model for gbm - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Chronic_GVHD.joblib
Saved threshold for gbm - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_Chronic_GVHD_threshold.json
[Split 70_30] gbm - Chronic_GVHD: test_auc=0.572, cv_auc_mean=0.562, SMOTE=True
Saved model for rf - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_Chronic_GVHD.joblib
Saved threshold for rf - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_Chronic_GVHD_threshold.json
[Split 70_30] rf - Chronic_GVHD: test_auc=0.548, cv_auc_mean=0.557, SMOTE=True
Saved model for svm - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_Chronic_GVHD.joblib
Saved threshold for svm - Chronic_GVHD -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_Chronic_GVHD_threshold.json
[Split 70_30] svm - Chronic_GVHD: test_auc=0.581, cv_auc_mean=0.559, SMOTE=True
[Split 80_20] xgb - Chronic_GVHD: test_auc=0.510, cv_auc_mean=0.587, SMOTE=False
[Split 80_20] gbm - Chronic_GVHD: test_auc=0.516, cv_auc_mean=0.583, SMOTE=True
[Split 80_20] rf - Chronic_GVHD: test_auc=0.516, cv_auc_mean=0.573, SMOTE=True
[Split 80_20] svm - Chronic_GVHD: test_auc=0.543, cv_auc_mean=0.575, SMOTE=True
Saved model for xgb - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_1_Year_Survival.joblib
Saved threshold for xgb - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/xgb_1_Year_Survival_threshold.json
[Split 70_30] xgb - 1_Year_Survival: test_auc=0.542, cv_auc_mean=0.549, SMOTE=False
Saved model for gbm - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_1_Year_Survival.joblib
Saved threshold for gbm - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/gbm_1_Year_Survival_threshold.json
[Split 70_30] gbm - 1_Year_Survival: test_auc=0.546, cv_auc_mean=0.554, SMOTE=True
Saved model for rf - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_1_Year_Survival.joblib
Saved threshold for rf - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/rf_1_Year_Survival_threshold.json
[Split 70_30] rf - 1_Year_Survival: test_auc=0.549, cv_auc_mean=0.555, SMOTE=True
Saved model for svm - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_1_Year_Survival.joblib
Saved threshold for svm - 1_Year_Survival -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/models/svm_1_Year_Survival_threshold.json
[Split 70_30] svm - 1_Year_Survival: test_auc=0.529, cv_auc_mean=0.561, SMOTE=True
[Split 80_20] xgb - 1_Year_Survival: test_auc=0.537, cv_auc_mean=0.558, SMOTE=False
[Split 80_20] gbm - 1_Year_Survival: test_auc=0.533, cv_auc_mean=0.561, SMOTE=True
[Split 80_20] rf - 1_Year_Survival: test_auc=0.546, cv_auc_mean=0.558, SMOTE=True
[Split 80_20] svm - 1_Year_Survival: test_auc=0.530, cv_auc_mean=0.565, SMOTE=True
Validation Summary (3-fold CV, 70/30 and 80/20 splits):
outcome	split	model	cv_auc_mean	cv_auc_std	cv_acc_mean	test_auc	test_acc	test_prec	test_rec	used_smote	auc_pass	auc_flag
25	1_Year_Survival	70_30	gbm	0.553838	0.002384	0.563268	0.546061	0.537723	0.539515	0.770214	True	False	FAIL
26	1_Year_Survival	70_30	rf	0.555391	0.002873	0.560448	0.549222	0.537331	0.553791	0.576909	True	False	FAIL
27	1_Year_Survival	70_30	svm	0.560700	0.001895	0.557428	0.529113	0.520870	0.529673	0.718315	True	False	FAIL
24	1_Year_Survival	70_30	xgb	0.548890	0.001750	0.562193	0.542319	0.519302	0.554555	0.393757	False	False	FAIL
29	1_Year_Survival	80_20	gbm	0.561000	0.004899	0.560819	0.533256	0.529062	0.560606	0.622372	True	False	FAIL
30	1_Year_Survival	80_20	rf	0.558231	0.001885	0.555487	0.545891	0.531808	0.575730	0.530698	True	False	FAIL
31	1_Year_Survival	80_20	svm	0.565077	0.001530	0.555712	0.530044	0.521739	0.563717	0.535744	True	False	FAIL
28	1_Year_Survival	80_20	xgb	0.557717	0.001380	0.559304	0.536895	0.531808	0.565149	0.605551	False	False	FAIL
17	Chronic_GVHD	70_30	gbm	0.562301	0.002035	0.747332	0.572210	0.647462	0.236387	0.378074	True	False	FAIL
18	Chronic_GVHD	70_30	rf	0.557362	0.008374	0.729208	0.547900	0.726631	0.239103	0.196721	True	False	FAIL
19	Chronic_GVHD	70_30	svm	0.558845	0.002578	0.696315	0.580646	0.730551	0.263345	0.227459	True	False	FAIL
16	Chronic_GVHD	70_30	xgb	0.556823	0.004813	0.760086	0.567497	0.642367	0.236172	0.389344	False	False	FAIL
21	Chronic_GVHD	80_20	gbm	0.583259	0.005051	0.754926	0.515830	0.566133	0.184845	0.403509	True	False	FAIL
22	Chronic_GVHD	80_20	rf	0.573164	0.004699	0.734942	0.516233	0.680092	0.176724	0.205514	True	False	FAIL
23	Chronic_GVHD	80_20	svm	0.575076	0.000978	0.702049	0.543464	0.648055	0.216258	0.353383	True	False	FAIL
20	Chronic_GVHD	80_20	xgb	0.587112	0.002513	0.767387	0.510099	0.675057	0.195695	0.250627	False	False	FAIL
1	Neutrophil_Engraftment	70_30	gbm	0.609658	0.006709	0.760220	0.560360	0.546933	0.745247	0.544142	True	False	FAIL
2	Neutrophil_Engraftment	70_30	rf	0.598387	0.005864	0.733638	0.551600	0.592985	0.724067	0.684064	True	False	FAIL
3	Neutrophil_Engraftment	70_30	svm	0.602295	0.008095	0.709203	0.545048	0.679600	0.716487	0.903665	True	False	FAIL
0	Neutrophil_Engraftment	70_30	xgb	0.605842	0.002581	0.770894	0.554438	0.479522	0.746611	0.397557	False	False	FAIL
5	Neutrophil_Engraftment	80_20	gbm	0.601097	0.005215	0.747684	0.579385	0.648513	0.741033	0.780910	True	False	FAIL
6	Neutrophil_Engraftment	80_20	rf	0.594091	0.005342	0.714230	0.552407	0.590847	0.738044	0.662396	True	False	FAIL
7	Neutrophil_Engraftment	80_20	svm	0.601157	0.003630	0.696267	0.538734	0.620137	0.730309	0.742473	True	False	FAIL
4	Neutrophil_Engraftment	80_20	xgb	0.606837	0.006078	0.756946	0.576609	0.626545	0.739242	0.737348	False	False	FAIL
9	Platelet_Engraftment	70_30	gbm	0.510895	0.003756	0.697993	0.544431	0.396042	0.756592	0.207915	True	False	FAIL
10	Platelet_Engraftment	70_30	rf	0.501881	0.003290	0.673962	0.529334	0.433471	0.727927	0.310201	True	False	FAIL
11	Platelet_Engraftment	70_30	svm	0.516376	0.007166	0.634289	0.552954	0.655693	0.713652	0.852285	True	False	FAIL
8	Platelet_Engraftment	70_30	xgb	0.512588	0.005822	0.709471	0.526899	0.430923	0.730148	0.302397	False	False	FAIL
13	Platelet_Engraftment	80_20	gbm	0.519370	0.008329	0.698288	0.548889	0.552860	0.747253	0.565941	True	False	FAIL
14	Platelet_Engraftment	80_20	rf	0.509139	0.010821	0.668593	0.547251	0.321281	0.776224	0.071063	True	False	FAIL
15	Platelet_Engraftment	80_20	svm	0.517063	0.002874	0.625259	0.561209	0.628375	0.732054	0.757362	True	False	FAIL
12	Platelet_Engraftment	80_20	xgb	0.525106	0.003548	0.705361	0.540951	0.445767	0.762332	0.326504	False	False	FAIL
Saved CV summary -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/metrics/cv_summary.csv
Best Models per Outcome (70/30 split):
/var/folders/3b/hnsfxtfj2w39h1fm21zknsf40000gn/T/ipykernel_35043/4048045446.py:322: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_models = cv_table[cv_table['split'] == "70_30"].groupby('outcome').apply(
outcome	model	test_auc	auc_flag	used_smote
0	1_Year_Survival	rf	0.549222	FAIL	True
1	Chronic_GVHD	svm	0.580646	FAIL	True
2	Neutrophil_Engraftment	gbm	0.560360	FAIL	True
3	Platelet_Engraftment	svm	0.552954	FAIL	True
Saved best models -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/metrics/best_models.csv
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_xgb_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_gbm_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_rf_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_svm_Neutrophil_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_xgb_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_gbm_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_rf_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_svm_Platelet_Engraftment.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_xgb_Chronic_GVHD.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_gbm_Chronic_GVHD.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_rf_Chronic_GVHD.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_svm_Chronic_GVHD.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_xgb_1_Year_Survival.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_gbm_1_Year_Survival.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_rf_1_Year_Survival.png
Saved ROC -> /Users/amanda/Desktop/UCBT/models_output_synthetic_best/figs/roc_svm_1_Year_Survival.png